In [1]:
# GPU check
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


In [2]:
# imports

# poetry add faiss-gpu # n.b. fails
# poetry add faiss-cpu
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import re

In [3]:
# load embedding models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Embeddings for semantic search
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)

# Retrieval model (E5)
RAG_MODEL = "intfloat/e5-base"
rag_model = SentenceTransformer(RAG_MODEL, device=DEVICE)

In [4]:
# Sample NHS‑Style Clinical Notes
clinical_notes = [
    "Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. ECG shows ST elevation.",
    "Patient reports mild headache and fatigue. Observations within normal range. Discharged with advice to rest.",
    "Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.",
    "Elderly patient with history of COPD, increased breathlessness over 3 days, using inhaler more frequently.",
    "Patient attended A&E following a fall. X-ray confirms fractured wrist. Plaster applied, follow-up in fracture clinic.",
    "Patient with abdominal pain, nausea, and vomiting. CT scan suggests possible appendicitis.",
    "Patient experiencing dizziness and blurred vision. Blood pressure elevated. Possible hypertensive episode."
]

pd.DataFrame({"note": clinical_notes})


,note
0,Patient presents with chest pain radiating to ...
1,Patient reports mild headache and fatigue. Obs...
2,Post-operative patient with fever and elevated...
3,"Elderly patient with history of COPD, increase..."
4,Patient attended A&E following a fall. X-ray c...
5,"Patient with abdominal pain, nausea, and vomit..."
6,Patient experiencing dizziness and blurred vis...


In [5]:
# Cleaning Function
def clean_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

cleaned_notes = [clean_text(t) for t in clinical_notes]

In [6]:
# Encode Notes (E5 for RAG)
rag_embeddings = rag_model.encode(cleaned_notes, convert_to_tensor=False)
rag_embeddings = np.array(rag_embeddings).astype("float32")
rag_embeddings.shape


(7, 768)

In [7]:
#  Build FAISS Index - CPU
d = rag_embeddings.shape[1]  # embedding dimension

# CPU FAISS index (works with faiss-cpu)
index = faiss.IndexFlatIP(d)

index.add(rag_embeddings)
print("Index size:", index.ntotal)


Index size: 7


#### Build FAISS Index - GPU n.b. If faiss-gpu is installed run this instead of previous
d = rag_embeddings.shape[1]  # embedding dimension

if torch.cuda.is_available():
    res = faiss.StandardGpuResources()
    index = faiss.GpuIndexFlatIP(res, d)  # inner product (cosine similarity)
else:
    index = faiss.IndexFlatIP(d)

index.add(rag_embeddings)
print("Index size:", index.ntotal)


In [8]:
# RAG Query Function
def rag_search(query: str, top_k: int = 3):
    q_emb = rag_model.encode(query, convert_to_tensor=False)
    q_emb = np.array([q_emb]).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({
            "note": cleaned_notes[idx],
            "score": float(score)
        })
    return pd.DataFrame(results)


In [9]:
# Test RAG Queries
queries = [
    "infection after surgery",
    "breathlessness COPD",
    "fracture wrist fall",
    "hypertensive episode dizziness"
]

for q in queries:
    print("\nQuery:", q)
    display(rag_search(q, top_k=3))



Query: infection after surgery


,note,score
0,Post-operative patient with fever and elevated...,0.861330
1,"Patient with abdominal pain, nausea, and vomit...",0.777528
2,Patient attended A&E following a fall. X-ray c...,0.758739



Query: breathlessness COPD


,note,score
0,"Elderly patient with history of COPD, increase...",0.892248
1,Patient presents with chest pain radiating to ...,0.813393
2,Patient reports mild headache and fatigue. Obs...,0.779431



Query: fracture wrist fall


,note,score
0,Patient attended A&E following a fall. X-ray c...,0.886705
1,Patient presents with chest pain radiating to ...,0.752031
2,Post-operative patient with fever and elevated...,0.750493



Query: hypertensive episode dizziness


,note,score
0,Patient experiencing dizziness and blurred vis...,0.903349
1,Patient presents with chest pain radiating to ...,0.823302
2,Patient reports mild headache and fatigue. Obs...,0.812086


In [10]:
# Add MiniLM Similarity - optional
mini_embeddings = emb_model.encode(cleaned_notes, convert_to_tensor=True)

def semantic_search(query: str, top_k: int = 3):
    q_emb = emb_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, mini_embeddings)[0]
    top_results = torch.topk(scores, k=top_k)

    results = []
    for idx, score in zip(top_results.indices, top_results.values):
        results.append({
            "note": cleaned_notes[idx],
            "score": float(score)
        })
    return pd.DataFrame(results)


In [11]:
# compare RAG vs semantic search
query = "infection after surgery"

print("RAG (E5):")
display(rag_search(query))

print("\nSemantic Search (MiniLM):")
display(semantic_search(query))


RAG (E5):


,note,score
0,Post-operative patient with fever and elevated...,0.861330
1,"Patient with abdominal pain, nausea, and vomit...",0.777528
2,Patient attended A&E following a fall. X-ray c...,0.758739



Semantic Search (MiniLM):


,note,score
0,Post-operative patient with fever and elevated...,0.555176
1,"Patient with abdominal pain, nausea, and vomit...",0.302507
2,Patient attended A&E following a fall. X-ray c...,0.240975


In [12]:
# Full RAG Pipeline Function
def rag_pipeline(query: str, top_k: int = 3):
    return {
        "query": query,
        "rag_results": rag_search(query, top_k),
        "semantic_results": semantic_search(query, top_k)
    }

result = rag_pipeline("breathlessness COPD", top_k=3)
result


{'query': 'breathlessness COPD',
 'rag_results':                                                 note     score
 0  Elderly patient with history of COPD, increase...  0.892248
 1  Patient presents with chest pain radiating to ...  0.813393
 2  Patient reports mild headache and fatigue. Obs...  0.779431,
 'semantic_results':                                                 note     score
 0  Elderly patient with history of COPD, increase...  0.712799
 1  Patient presents with chest pain radiating to ...  0.385078
 2  Patient experiencing dizziness and blurred vis...  0.314779}

In [13]:
print("Query:", result["query"])

print("\nRAG Results (E5):")
display(result["rag_results"])

print("\nSemantic Search Results (MiniLM):")
display(result["semantic_results"])


Query: breathlessness COPD

RAG Results (E5):


,note,score
0,"Elderly patient with history of COPD, increase...",0.892248
1,Patient presents with chest pain radiating to ...,0.813393
2,Patient reports mild headache and fatigue. Obs...,0.779431



Semantic Search Results (MiniLM):


,note,score
0,"Elderly patient with history of COPD, increase...",0.712799
1,Patient presents with chest pain radiating to ...,0.385078
2,Patient experiencing dizziness and blurred vis...,0.314779


Why RAG (E5) scores are higher:
- E5 is trained specifically for retrieval
- E5 embeddings are perfectly normalised
- E5 uses query/document prefixes
- E5’s training objective pushes relevant pairs to high similarity
- FAISS inner‑product search amplifies E5’s alignment
- MiniLM is general‑purpose, not retrieval‑optimised